# Day 5: Embeddings + Vector DB with Chroma

## Goal

Create embeddings for documents, store them in Chroma, and run similarity search.

By the end of this notebook you will have:

- generated embeddings for sample documents
- stored those vectors in Chroma
- queried the collection with a natural-language question
- retrieved the most similar documents

This is the first building block of a basic RAG system.

## Step 0: Sync the Project Once

The Day 5 dependencies are already included in `pyproject.toml`, so learners only need:

```bash
uv sync
```

Set these values in `.env`:

```text
OPENAI_API_KEY=your_openai_key
OPENAI_EMBEDDING_MODEL=text-embedding-3-small
```

For course maintainers, the Day 5 packages were added with:

```bash
uv add "chromadb==0.5.23" "onnxruntime==1.20.1"
```

This repo uses a Chroma-compatible dependency set that works on the current project environment.

In [ ]:
import os
from pathlib import Path

import chromadb
from dotenv import load_dotenv
from openai import OpenAI

print("Imports loaded successfully.")

In [ ]:
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

{
    "OPENAI_API_KEY_present": bool(OPENAI_API_KEY),
    "OPENAI_EMBEDDING_MODEL": EMBEDDING_MODEL,
}

## Step 1: Create a Small Document Set

We will use a few short documents so it is easy to see what similarity search is doing.

In [ ]:
documents = [
    {
        "id": "doc-1",
        "title": "Invoice Processing",
        "text": "Invoices usually contain vendor name, invoice number, invoice date, due date, line items, taxes, and total amount.",
    },
    {
        "id": "doc-2",
        "title": "Leave Policy",
        "text": "Employees can apply for annual leave through the HR portal. Manager approval is required before leave is confirmed.",
    },
    {
        "id": "doc-3",
        "title": "Travel Reimbursement",
        "text": "Travel reimbursement claims should include receipts, project code, travel dates, and approval from the reporting manager.",
    },
    {
        "id": "doc-4",
        "title": "Vendor Payments",
        "text": "Accounts payable teams verify invoice totals, validate bank details, and schedule vendor payments based on due dates.",
    },
]

documents

## Step 2: Generate Embeddings

An embedding is a numeric vector representation of text.

Texts with similar meaning tend to end up closer together in vector space.

In [ ]:
def get_openai_client() -> OpenAI:
    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY is not set in .env")
    return OpenAI(api_key=OPENAI_API_KEY)


def embed_texts(texts: list[str], model: str) -> list[list[float]]:
    client = get_openai_client()
    response = client.embeddings.create(model=model, input=texts)
    return [item.embedding for item in response.data]


print("Embedding helpers ready.")

In [ ]:
texts = [doc["text"] for doc in documents]

if not OPENAI_API_KEY:
    print("Set OPENAI_API_KEY in .env before generating embeddings.")
else:
    embeddings = embed_texts(texts, EMBEDDING_MODEL)
    print(f"Generated {len(embeddings)} embeddings.")
    print(f"Vector size for the first embedding: {len(embeddings[0])}")

## Step 3: Store the Vectors in Chroma

We will use a local persistent Chroma database stored inside the project so learners can inspect it later.

In [ ]:
db_path = Path("day5/chroma_store")
db_path.mkdir(parents=True, exist_ok=True)

# Disable anonymized telemetry to avoid noisy telemetry errors in some environments.
settings = chromadb.Settings(anonymized_telemetry=False)
chroma_client = chromadb.PersistentClient(path=str(db_path), settings=settings)
collection_name = "training_day5_documents"

try:
    chroma_client.delete_collection(collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(name=collection_name)
print(f"Collection ready: {collection_name}")


In [ ]:
if not OPENAI_API_KEY:
    print("Skipping Chroma insert because embeddings were not generated.")
else:
    collection.add(
        ids=[doc["id"] for doc in documents],
        documents=[doc["text"] for doc in documents],
        metadatas=[{"title": doc["title"]} for doc in documents],
        embeddings=embeddings,
    )
    print(f"Inserted {collection.count()} documents into Chroma.")

## Step 4: Query by Similarity

Now we embed the user question, send that vector to Chroma, and ask for the nearest documents.

In [ ]:
query = "Which documents talk about invoice totals and vendor payments?"
query

In [ ]:
if not OPENAI_API_KEY:
    print("Set OPENAI_API_KEY in .env before running similarity search.")
else:
    query_embedding = embed_texts([query], EMBEDDING_MODEL)[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=2,
    )

    results

In [ ]:
if not OPENAI_API_KEY:
    print("Set OPENAI_API_KEY in .env before inspecting similarity results.")
else:
    for i, doc_id in enumerate(results["ids"][0], start=1):
        title = results["metadatas"][0][i - 1]["title"]
        document = results["documents"][0][i - 1]
        distance = results["distances"][0][i - 1]

        print(f"Match {i}")
        print("ID:", doc_id)
        print("Title:", title)
        print("Distance:", distance)
        print("Text:", document)
        print("-" * 80)

## Step 5: Try Another Query

Change the text below and run the next two cells again.

Examples:

- `How do employees request leave?`
- `Which document is about manager approval for travel expenses?`
- `Find content related to invoice due dates.`

In [ ]:
query = "How do employees request leave?"
query

In [ ]:
if not OPENAI_API_KEY:
    print("Set OPENAI_API_KEY in .env before running similarity search.")
else:
    query_embedding = embed_texts([query], EMBEDDING_MODEL)[0]
    results = collection.query(query_embeddings=[query_embedding], n_results=2)

    for i, doc_id in enumerate(results["ids"][0], start=1):
        print(f"Match {i}: {doc_id} | {results['metadatas'][0][i - 1]['title']}")
        print(results["documents"][0][i - 1])
        print("-" * 80)

## Day 5 Recap

You now have a working similarity-search pipeline:

- documents were converted into embeddings
- vectors were stored in Chroma
- a user query was embedded the same way
- Chroma returned the nearest matching documents

This is the core of retrieval before you add an LLM answer generation step on top.